# Exercise 15: Sim-to-Real Perturbation Suite

This notebook demonstrates a perturbation matrix using the reaching proxy. The same reporting pattern applies to quadruped locomotion.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from ch13_guide.envs.reaching_env import ReachingConfig, ReachingEnv
from ch13_guide.utils import proportional_controller, rollout

In [ ]:
scenarios = {
    "nominal": ReachingConfig(),
    "observation_noise": ReachingConfig(observation_noise_std=0.04),
    "action_noise": ReachingConfig(action_noise_std=0.02),
    "one_step_delay": ReachingConfig(action_delay_steps=1),
    "two_step_delay": ReachingConfig(action_delay_steps=2),
    "reduced_actuation": ReachingConfig(max_action=0.05),
    "larger_unsafe_region": ReachingConfig(unsafe_radius=0.28),
}

rows = []
for name, config in scenarios.items():
    results = rollout(
        ReachingEnv(config),
        proportional_controller,
        episodes=100,
        seed=100,
    )
    for record in results:
        rows.append({"scenario": name, **record})

df = pd.DataFrame(rows)
summary = df.groupby("scenario").agg(
    success_rate=("success", "mean"),
    mean_steps=("steps", "mean"),
    final_distance=("final_distance", "mean"),
    interventions=("safety_interventions", "mean"),
)
summary

In [ ]:
nominal_success = summary.loc["nominal", "success_rate"]
summary["passes_example_rule"] = (
    summary["success_rate"] >= 0.90 * nominal_success
)
summary

## Quadruped extension

Replace the proxy perturbations with:

- body mass and inertia;
- ground friction;
- actuator strength;
- command delay and jitter;
- IMU and encoder noise;
- terrain geometry;
- external pushes.

Predeclare pass/fail thresholds before viewing results.